# 2. Application to Disease Specific Genes

This notebook will apply the previously defined function to perform *in silico* KO/KI of genes previously associated with ALS.
The focus will be KO/KI of genes previously linked to ALS globally across a single cell type (excitory) in the data set.

Specifically aiming to identify genes protective of ALS by KO of the genes in healthy cells and KI of the genes in ALS cells.

Effects within additional cell subtypes and combinatorial testing of genes are not considered but would be a good next step.

## Setup

First the functions and data are loaded from the previous notebook.

In [1]:
from helical.models.geneformer import Geneformer, GeneformerConfig
import anndata as ad
import scanpy as sc
import random
import numpy as np
import pickle

#load the model
model_config = GeneformerConfig(model_name="gf-12L-38M-i4096", batch_size=10)
geneformer = Geneformer(model_config)

#Load the data to test
ann_data = ad.read_h5ad("counts_combined_filtered_BA4_sALS_PN.h5ad")

#Add a metadata combining disease and cell type
ann_data.obs["ClassDis"] = [x[0]+"_"+x[1] for x in zip(list(ann_data.obs["CellClass"]), list(ann_data.obs["Condition"]))]

#set seed for reproducibility
random.seed(100)

def perturb(ann_dat, mod, cellfeature="", featuredef="", numcells=10, genes=["PDCD1"], direction="KO"):
    #id the cell types of interest
    targcells =  ann_data[ann_data.obs[cellfeature] == featuredef]
    #randomly subsample x cells
    subsamp = random.sample(range(len(targcells)),min(len(targcells),numcells))
    #subset adat
    sub_adat = targcells[subsamp]
    sc.pp.filter_genes(sub_adat, min_cells=1)
    #check if target gene is present and give error if not for KO
    if any(x not in sub_adat.var_names for x in genes) and direction == "KO":
        print("At least one gene not seen in any sampled cells.")
        return
    sums = sum(sub_adat.X[:, sub_adat.var_names.get_indexer(genes)])
    if sums.max()==0:
        print("Gene have zero expression in all sampled cells already.")
        return()
    #modify to ki or ko gene
    mod_adat = sub_adat.copy()
    if direction == "KO":
        mod_adat.X[:, mod_adat.var_names.get_indexer(genes)] = 0
    elif direction == "KI":
        mod_adat.X[:, mod_adat.var_names.get_indexer(genes)] = mod_adat.X.max()
    #generate the embeddings
    baseem = geneformer.get_embeddings(geneformer.process_data(sub_adat))
    modem = geneformer.get_embeddings(geneformer.process_data(mod_adat))
    embeds = {"base":baseem,"modified":modem,"params":[cellfeature,featuredef,numcells,genes,direction]}
    return(embeds)


2026-02-18 14:24:29,989 - WARNING:py.warnings:/home/mjacksonwood/workspace/conda/envs/helical-package/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

2026-02-18 14:24:30,223 - INFO:datasets:PyTorch version 2.7.0 available.
2026-02-18 14:24:43,500 - INFO:helical.models.geneformer.model:Model finished initializing.
2026-02-18 14:24:43,500 - INFO:helical.models.geneformer.model:'gf-12L-38M-i4096' model is in 'eval' mode, on device 'cpu' with embedding mode 'cell'.


## Define disease associted genes

ALS associated genes were taken as the rare variants with a high effect size from a recent review by [Nijs et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC11377058/). This provides a list of genes expected to have a large effect but a small enough set to run in a reasonable amount of time given available hardware.

These are: C9orf72, SOD1, TARDBP, FUS, TBK1, TUBA4A, UBQLN2, VCP, OPTN, and NEK1

In [2]:
#check these genes are in the data set
genes = ["C9orf72","SOD1","TARDBP","FUS","TBK1","TUBA4A","UBQLN2","VCP","OPTN","NEK1"]
print([x in ann_data.var_names for x in genes])

[True, True, True, True, True, True, True, True, True, True]


## Run the KO and KI and generate embeddings

For each gene embeddings are generated from either healthy excitory cells (PN = Pathologically normal) with the gene KO or ALS excitory cells with the gene KI.

20 cells are sampled for each gene and the function returns the cell embeddings before and after the modification.

These embeddings are saved for downstream analyses to infer the biological consequences of the modelled modifications.

In [3]:
#run the modification and embedding function
healthyko = {x:perturb(ann_data, geneformer, cellfeature="ClassDis", featuredef="Ex_PN", numcells=20, genes=[x], direction="KO") for x in genes}
alski = {x:perturb(ann_data, geneformer, cellfeature="ClassDis", featuredef="Ex_ALS", numcells=20, genes=[x], direction="KI") for x in genes}


2026-02-18 14:24:54,949 - WARNING:py.warnings:/home/mjacksonwood/workspace/conda/envs/helical-package/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:293: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number

2026-02-18 14:24:54,974 - WARNING:py.warnings:/home/mjacksonwood/workspace/conda/envs/helical-package/lib/python3.11/site-packages/scipy/sparse/_index.py:151: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)

2026-02-18 14:24:54,976 - INFO:helical.models.geneformer.model:Processing data for Geneformer.
2026-02-18 14:24:55,499 - INFO:helical.utils.mapping:Mapped 14065 / 14086 genes to Ensembl IDs.
2026-02-18 14:24:55,537 - INFO:helical.models.geneformer.geneformer_tokenizer:AnnData object with n_obs × n_vars = 20 × 14086
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', '

In [4]:
#save the outputs
allemb = {"KO":healthyko, "KI":alski}
outfile = open("mod_embed.pkl", 'wb') 
pickle.dump(allemb, outfile)
outfile.close()